[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/concept_notebooks/Learning_Concepts_Deep_Dive.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/concept_notebooks/Learning_Concepts_Deep_Dive.ipynb)

# How Learning Happens — From One Observation to Complex Functions

**What this notebook answers**

1. For each observation, exactly how does it push each weight in a direction?
2. When many observations disagree, how does learning settle on the right weights?
3. What does the "loss surface" look like — and how does gradient descent navigate it?
4. Where does a linear function fail — and why do we need complex functions?
5. What IS a complex function — how does stacking layers create expressive power?
6. How does a neural network use the chain rule to update weights in early layers?
7. What do hidden layers actually learn — what intermediate features do they invent?

---

## Section 1 — One Observation, One Vote on Each Weight

### Recap: the gradient for one observation

For observation `i` with features `[x₁, x₂]` and target `y`:

```
prediction  ŷ  =  w₁·x₁  +  w₂·x₂  +  b
error        e  =  ŷ  −  y

gradient for w₁  =  e × x₁      ← "how much did w₁ cause this error?"
gradient for w₂  =  e × x₂
gradient for b   =  e × 1

update:  w₁ ← w₁ − lr × (e × x₁)
```

### Why multiply error by the feature value?

The feature value `x₁` is a **lever**.

- If the model over-predicted (`e > 0`) and `x₁` was large:
  → w₁ pushed the prediction up too much → reduce w₁
- If the model over-predicted (`e > 0`) but `x₁ ≈ 0`:
  → w₁ barely affected this prediction → almost no change to w₁
- If `x₁` was negative and error was positive:
  → w₁ actually *pulled* the prediction down → increase w₁

**Each observation casts a "vote" on the direction of each weight.**
The vote's size is proportional to how involved that feature was in the prediction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

# ── Build a tiny 2-feature dataset ───────────────────────────────────────────
N = 80
x1 = np.random.uniform(0, 10, N)
x2 = np.random.uniform(0, 10, N)
TRUE_W1, TRUE_W2, TRUE_B = 3.0, -1.5, 5.0
y = TRUE_W1 * x1 + TRUE_W2 * x2 + TRUE_B + np.random.normal(0, 2, N)

X = np.column_stack([x1, x2])

# ── Show the per-observation vote ─────────────────────────────────────────────
w1, w2, b = 0.0, 0.0, 0.0   # start from zero

print("First 8 observations — vote each casts on w1 and w2")
print(f"{'Obs':>4}  {'x1':>6}  {'x2':>6}  {'y':>7}  {'ŷ':>7}  {'error':>7}  "
      f"{'vote w1':>9}  {'vote w2':>9}  {'action on w1'}")
print("-" * 90)
for i in range(8):
    y_hat  = w1*x1[i] + w2*x2[i] + b
    error  = y_hat - y[i]
    vote1  = error * x1[i]
    vote2  = error * x2[i]
    action = "decrease w1 ↓" if vote1 > 0 else "increase w1 ↑"
    print(f"{i:>4}  {x1[i]:>6.2f}  {x2[i]:>6.2f}  {y[i]:>7.2f}  {y_hat:>7.2f}  "
          f"{error:>7.2f}  {vote1:>9.2f}  {vote2:>9.2f}  {action}")
print()
print("Each row = one observation's vote on which direction to move w1 and w2.")
print("Large |vote| = that observation strongly believes the weight should change.")

In [ ]:
# Visualise 10 individual votes on w1 and w2 as arrows in weight space
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = range(10)
w1_curr, w2_curr = 0.0, 0.0

arrow_colors = plt.cm.tab10(np.linspace(0, 1, len(sample)))

for ax_idx, (ax, feat_idx, feat_name, true_w) in enumerate(
        [(axes[0], 0, 'w1', TRUE_W1), (axes[1], 1, 'w2', TRUE_W2)]):

    w_curr = [0.0, 0.0]
    LR = 0.005

    for k, i in enumerate(sample):
        y_hat   = w_curr[0]*x1[i] + w_curr[1]*x2[i] + b
        error   = y_hat - y[i]
        votes   = [error * x1[i], error * x2[i]]
        new_w   = w_curr[feat_idx] - LR * votes[feat_idx]
        ax.annotate('', xy=(k+1, new_w), xytext=(k, w_curr[feat_idx]),
                    arrowprops=dict(arrowstyle='->', color=arrow_colors[k], lw=1.5))
        ax.scatter(k, w_curr[feat_idx], color=arrow_colors[k], s=40, zorder=5)
        w_curr[feat_idx] = new_w

    ax.axhline(true_w, color='red', ls='--', lw=1.5, label=f'True {feat_name}={true_w}')
    ax.set_xlabel('Observation number', fontsize=11)
    ax.set_ylabel(f'{feat_name} value', fontsize=11)
    ax.set_title(f'{feat_name}: Each Observation Nudges the Weight\n(arrows = individual votes)',
                 fontsize=11)
    ax.legend()

plt.suptitle('10 Observations Voting on w1 and w2 — Each Vote is One Gradient Step',
             fontsize=12)
plt.tight_layout()
plt.show()
print("The weight zigzags toward the true value as each observation votes.")
print("Some push the right way, some push the wrong way — the trend is toward the truth.")

## Section 2 — When Observations Disagree: The Loss Surface

### The weight space

Imagine every possible pair of weights `(w1, w2)` as a point on a 2D plane.
For each point, we can compute the total error (MSE) the model makes with those weights.

This creates a **3D surface** — the **loss surface** — where:
- The x and y axes are the weight values
- The z axis (height) is the MSE error
- The lowest point is where the best weights live

**Gradient descent is walking downhill on this surface.**

### Why the surface is bowl-shaped for linear regression

For linear regression, the MSE is a quadratic function of the weights:
```
MSE = (1/N) Σ (w₁x₁ᵢ + w₂x₂ᵢ + b − yᵢ)²
```
This is always a **convex bowl** — exactly one minimum.
No matter where you start, gradient descent will find it.

### Why this matters: complex functions do NOT have a clean bowl

For neural networks, the loss surface has:
- **Multiple valleys** (local minima)
- **Flat plateaus** (where gradient ≈ 0, but not the true minimum)
- **Saddle points** (gradient = 0, but going up in one direction and down in another)

This is why neural network training is harder and requires techniques like
momentum, learning rate schedules, and batch normalisation.

In [ ]:
# Plot the loss surface for our 2-feature linear model
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)
TRUE_W1_SC = TRUE_W1 * X[:, 0].std()
TRUE_W2_SC = TRUE_W2 * X[:, 1].std()

# Build loss surface over a grid of (w1, w2) values
w1_range = np.linspace(-2, 8, 80)
w2_range = np.linspace(-6, 4, 80)
W1, W2   = np.meshgrid(w1_range, w2_range)
MSE_GRID = np.array([
    np.mean((W1.ravel()[i] * X_sc[:,0] + W2.ravel()[i] * X_sc[:,1] - y)**2)
    for i in range(W1.size)
]).reshape(W1.shape)

fig = plt.figure(figsize=(16, 5))

# 3D surface
ax3d = fig.add_subplot(131, projection='3d')
ax3d.plot_surface(W1, W2, np.log10(MSE_GRID), cmap='viridis', alpha=0.85)
ax3d.set_xlabel('w1', fontsize=9)
ax3d.set_ylabel('w2', fontsize=9)
ax3d.set_zlabel('log10(MSE)', fontsize=9)
ax3d.set_title('Loss Surface (3D)\nConvex bowl — one minimum', fontsize=10)
ax3d.view_init(elev=25, azim=-60)

# Contour map
ax2 = fig.add_subplot(132)
cs = ax2.contourf(W1, W2, np.log10(MSE_GRID), levels=30, cmap='viridis')
plt.colorbar(cs, ax=ax2, label='log10(MSE)')
ax2.plot(TRUE_W1_SC, TRUE_W2_SC, 'r*', markersize=14, label='True weights')
ax2.set_xlabel('w1 (scaled space)', fontsize=10)
ax2.set_ylabel('w2 (scaled space)', fontsize=10)
ax2.set_title('Loss Surface (top view)\nDarker = lower error', fontsize=10)
ax2.legend()

# Gradient descent path on the contour
ax3 = fig.add_subplot(133)
cs2 = ax3.contourf(W1, W2, np.log10(MSE_GRID), levels=30, cmap='viridis')
ax3.plot(TRUE_W1_SC, TRUE_W2_SC, 'r*', markersize=14, label='True weights')

w1_gd, w2_gd, b_gd = 6.0, 3.0, 0.0
LR = 0.08
path_w1, path_w2 = [w1_gd], [w2_gd]
for _ in range(60):
    y_hat  = w1_gd * X_sc[:,0] + w2_gd * X_sc[:,1] + b_gd
    error  = y_hat - y
    w1_gd -= LR * (error * X_sc[:,0]).mean()
    w2_gd -= LR * (error * X_sc[:,1]).mean()
    b_gd  -= LR * error.mean()
    path_w1.append(w1_gd)
    path_w2.append(w2_gd)

ax3.plot(path_w1, path_w2, 'w-o', markersize=4, lw=1.5, alpha=0.9, label='GD path')
ax3.plot(path_w1[0], path_w2[0], 'wo', markersize=10, label='Start')
ax3.set_xlabel('w1 (scaled space)', fontsize=10)
ax3.set_ylabel('w2 (scaled space)', fontsize=10)
ax3.set_title('Gradient Descent Path\nWalking downhill to minimum', fontsize=10)
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Gradient descent spirals down the bowl toward the minimum (true weights).")

## Section 3 — Where a Linear Function Fails

### A linear function draws a flat plane through the data

In 2D: `y = w₁x₁ + b` is a straight line.
In 3D: `y = w₁x₁ + w₂x₂ + b` is a flat plane.
No matter how many features you add, the relationship is always: "a constant amount of y per unit of x."

### Real-world data is rarely linear

Consider these patterns a flat plane cannot capture:

| Pattern | Example | Why linear fails |
|---|---|---|
| Curve | House price levels off at very high income | Line keeps growing forever |
| Interaction | Drug effect depends on both dose AND patient weight | Line treats each feature independently |
| Threshold | Risk jumps sharply above a certain age | Line has no concept of a cliff |
| XOR | Satisfy conditions A OR B but not both | Impossible for any straight line |
| Oscillation | Seasonal sales patterns | A single slope cannot oscillate |

### Two strategies for complex patterns

**Strategy 1 — Polynomial/feature engineering**: manually create new features like x², x₁×x₂
The model is still linear in the new features, but captures non-linear patterns in the original space.

**Strategy 2 — Neural network**: automatically learn the right intermediate features.
The network discovers its own transformations of the input, composing them into complex functions.

In [ ]:
# Demonstrate: linear model fails on curved / XOR data

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Case 1: Curve ─────────────────────────────────────────────────────────────
x_curve = np.linspace(0, 10, 200)
y_curve = 2 * np.sin(x_curve) + 0.3 * x_curve + np.random.normal(0, 0.3, 200)
w_lin = np.polyfit(x_curve, y_curve, 1)
w_poly = np.polyfit(x_curve, y_curve, 4)

axes[0].scatter(x_curve, y_curve, s=8, alpha=0.5, color='steelblue', label='Data')
axes[0].plot(x_curve, np.polyval(w_lin, x_curve),  'r-', lw=2, label='Linear fit')
axes[0].plot(x_curve, np.polyval(w_poly, x_curve), 'g-', lw=2, label='Polynomial fit')
axes[0].set_title('Curved Data\nLinear line misses the pattern', fontsize=11)
axes[0].set_xlabel('x', fontsize=10)
axes[0].set_ylabel('y', fontsize=10)
axes[0].legend(fontsize=9)

# ── Case 2: Threshold ─────────────────────────────────────────────────────────
x_thresh = np.random.uniform(0, 10, 200)
y_thresh = (x_thresh > 6).astype(float) + np.random.normal(0, 0.1, 200)
w_t = np.polyfit(x_thresh, y_thresh, 1)

axes[1].scatter(x_thresh, y_thresh, s=8, alpha=0.4, color='#FFA726')
axes[1].plot(sorted(x_thresh), np.polyval(w_t, sorted(x_thresh)), 'r-', lw=2,
             label='Linear fit (fails at cliff)')
axes[1].axvline(6, color='green', ls='--', lw=2, label='True threshold')
axes[1].set_title('Threshold / Step Data\nLinear line misses the cliff', fontsize=11)
axes[1].set_xlabel('x', fontsize=10)
axes[1].set_ylabel('y', fontsize=10)
axes[1].legend(fontsize=9)

# ── Case 3: XOR ───────────────────────────────────────────────────────────────
x_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])
xs, ys = np.meshgrid(np.linspace(-0.5, 1.5, 80), np.linspace(-0.5, 1.5, 80))

axes[2].scatter(x_xor[:, 0], x_xor[:, 1], c=y_xor, cmap='RdYlGn',
                s=400, zorder=5, edgecolors='black')
for xi, yi, label in zip(x_xor, x_xor, ['0','1','1','0']):
    axes[2].text(xi[0]+0.05, xi[1]+0.05, label, fontsize=12, fontweight='bold')
axes[2].set_title('XOR Problem\nNo straight line can separate green from red', fontsize=11)
axes[2].set_xlabel('feature 1', fontsize=10)
axes[2].set_ylabel('feature 2', fontsize=10)
axes[2].set_xlim(-0.5, 1.5)
axes[2].set_ylim(-0.5, 1.5)
# Show that any linear boundary fails
axes[2].plot([-0.5, 1.5], [1.0, 0.0], 'r--', lw=1.5, label='Any line fails')
axes[2].legend(fontsize=9)

plt.suptitle('Three Patterns a Linear Function Cannot Learn', fontsize=12)
plt.tight_layout()
plt.show()

## Section 4 — What IS a Complex Function?

### Stacking simple operations creates complexity

A single linear function: `y = wx + b` — a straight line.

Add a non-linearity (activation function) to one transformed feature:
```
h  = relu(w₁x₁ + w₂x₂ + b₁)     ← hidden unit
y  = v·h + b₂                     ← output
```
`relu(z) = max(0, z)` — zero for negative inputs, identity for positive.

This simple addition already allows the model to learn a **bent line** (a hinge).

Add multiple hidden units — each learns a different hinge point:
```
h₁ = relu(w₁₁x₁ + w₁₂x₂ + b₁)    ← hinge at one location
h₂ = relu(w₂₁x₁ + w₂₂x₂ + b₂)    ← hinge at another location
h₃ = relu(w₃₁x₁ + w₃₂x₂ + b₃)    ← hinge at another location
y  = v₁h₁ + v₂h₂ + v₃h₃ + b_out  ← sum of hinges = any piecewise-linear curve
```

**The universal approximation theorem** says: with enough hidden units,
a single hidden layer can approximate ANY continuous function.

### The key ingredient: the activation function

Without a non-linear activation function, stacking linear layers is still just a linear function:
```
Layer 2 output = W₂(W₁x + b₁) + b₂ = (W₂W₁)x + (W₂b₁ + b₂) = W_new × x + b_new
```
Two linear layers collapse into one. The non-linearity (relu, sigmoid, tanh) is what makes
deeper networks genuinely more expressive than shallow ones.

In [ ]:
# Show how combining ReLU units builds complex functions
x_vis = np.linspace(-3, 3, 300)

def relu(z): return np.maximum(0, z)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

# Single linear unit
y_lin = 1.5 * x_vis + 0.5
axes[0].plot(x_vis, y_lin, 'steelblue', lw=2)
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Single linear unit\ny = 1.5x + 0.5', fontsize=10)
axes[0].set_ylim(-4, 5)

# Single ReLU unit — a hinge
y_relu = relu(1.5 * x_vis + 0.5)
axes[1].plot(x_vis, y_relu, '#26A69A', lw=2)
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Single ReLU unit\nrelu(1.5x + 0.5) — one hinge', fontsize=10)
axes[1].set_ylim(-4, 5)

# Two ReLU units added
h1 = relu(2.0 * x_vis - 1.0)
h2 = relu(-1.5 * x_vis + 0.5)
y_two = 1.2 * h1 - 0.8 * h2 + 0.3
axes[2].plot(x_vis, h1 * 1.2, '--', color='steelblue', alpha=0.5, lw=1.5, label='1.2 × h1')
axes[2].plot(x_vis, -h2 * 0.8, '--', color='#FFA726', alpha=0.5, lw=1.5, label='-0.8 × h2')
axes[2].plot(x_vis, y_two, '#AB47BC', lw=2.5, label='Combined')
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_title('Two ReLU units\ntwo hinges → one curve', fontsize=10)
axes[2].legend(fontsize=8)
axes[2].set_ylim(-4, 5)

# Six ReLU units — approximating a sine wave
np.random.seed(3)
n_units = 6
W_in  = np.random.randn(n_units) * 2
B_in  = np.random.randn(n_units) * 1.5
W_out = np.random.randn(n_units)

H     = relu(np.outer(x_vis, W_in) + B_in)
y_six = H @ W_out
y_six -= y_six.mean()

axes[3].plot(x_vis, y_six, '#EF5350', lw=2.5, label='6 ReLU units')
axes[3].axhline(0, color='black', lw=0.5)
axes[3].set_title('Six ReLU units\ncomplex piecewise curve', fontsize=10)
axes[3].set_ylim(-4, 5)

plt.suptitle('Stacking ReLU Units: Simple → Complex Function', fontsize=12)
for ax in axes: ax.set_xlabel('x', fontsize=10)
plt.tight_layout()
plt.show()
print("More ReLU units = more hinges = finer-grained approximation of any curve.")

## Section 5 — Forward Pass: How the Network Makes a Prediction

### A two-layer neural network

```
Input layer → Hidden layer → Output layer

x → [W₁, b₁] → relu(·) → h → [W₂, b₂] → sigmoid(·) → ŷ
```

For each observation `x`:

**Step 1 — Pre-activation of hidden layer:**
```
z₁ = W₁ · x + b₁         (weighted sum, same as linear regression)
```

**Step 2 — Activation of hidden layer:**
```
h = relu(z₁)              (apply non-linearity: set negatives to zero)
```

**Step 3 — Pre-activation of output layer:**
```
z₂ = W₂ · h + b₂         (another weighted sum — but inputs are h, not x)
```

**Step 4 — Output:**
```
ŷ = sigmoid(z₂)          (for classification: squash to probability 0–1)
```

The hidden layer `h` is the key: it is a **new representation of the input**,
learned by the network, that makes the final prediction easier.
The network does not use `x` directly to predict — it uses `h`, which it designed itself.

In [ ]:
# Forward pass in transparent code — solve the XOR problem
# XOR: output=1 when exactly one input is 1, output=0 otherwise
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

# Hand-crafted weights that solve XOR (to show the concept clearly)
W1 = np.array([[ 1.,  1.],
               [ 1.,  1.]])   # shape (2 hidden, 2 inputs)
b1 = np.array([ 0., -1.])     # bias for each hidden unit

W2 = np.array([[ 1., -2.]])   # shape (1 output, 2 hidden)
b2 = np.array([0.])

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

print("Forward pass — XOR network")
print("=" * 75)
print(f"{'Input':>12}  {'z1 = W1x+b1':>22}  {'h = relu(z1)':>20}  "
      f"{'z2 = W2h+b2':>14}  {'ŷ':>6}  {'true':>5}")
print("-" * 75)
for xi, yi in zip(X_xor, y_xor):
    z1   = W1 @ xi + b1
    h    = relu(z1)
    z2   = W2 @ h  + b2
    yhat = sigmoid(z2)
    print(f"  {str(xi.astype(int)):>10}  {str(z1.round(1)):>22}  {str(h.round(1)):>20}  "
          f"{z2[0]:>14.2f}  {yhat[0]:>6.3f}  {int(yi):>5}")

print()
print("The hidden layer h transforms the inputs into a NEW representation.")
print("In the h-space, the XOR classes BECOME linearly separable.")

In [ ]:
# Visualize: XOR is NOT linearly separable in input space, but IS in hidden space
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_xor = ['#EF5350' if y == 0 else '#26A69A' for y in y_xor]
labels_xor = ['0 (red)' if y == 0 else '1 (green)' for y in y_xor]

# Input space
for xi, c, lbl in zip(X_xor, colors_xor, labels_xor):
    axes[0].scatter(xi[0], xi[1], color=c, s=300, zorder=5)
    axes[0].annotate(f"({xi[0]:.0f},{xi[1]:.0f})\ny={lbl.split()[0]}",
                     (xi[0], xi[1]), textcoords='offset points', xytext=(10, 5), fontsize=9)
axes[0].set_xlim(-0.5, 1.8); axes[0].set_ylim(-0.5, 1.8)
axes[0].set_title('Input Space (x1, x2)\nNOT linearly separable\n(no line can split red from green)',
                  fontsize=10)
axes[0].set_xlabel('x1'); axes[0].set_ylabel('x2')

# Hidden space — after first layer
h_all = relu(W1 @ X_xor.T + b1[:, None]).T   # shape (4, 2)
for hi, c, lbl in zip(h_all, colors_xor, labels_xor):
    axes[1].scatter(hi[0], hi[1], color=c, s=300, zorder=5)
    axes[1].annotate(f"({hi[0]:.1f},{hi[1]:.1f})\ny={lbl.split()[0]}",
                     (hi[0], hi[1]), textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[1].set_title('Hidden Space (h1, h2) — after relu(W1x + b1)\nNOW linearly separable!\n'
                  'Network invented a new representation', fontsize=10)
axes[1].set_xlabel('h1 (hidden unit 1)'); axes[1].set_ylabel('h2 (hidden unit 2)')
x_sep = np.linspace(-0.2, 1.2, 50)
axes[1].plot(x_sep, 0.5 * x_sep + 0.2, 'k--', lw=2, label='Linear boundary now works')
axes[1].legend(fontsize=9)

# Show what the two hidden units compute individually
x_grid = np.linspace(-0.2, 1.2, 100)
axes[2].plot(x_grid, relu(W1[0, 0] * x_grid + b1[0]), '#5C6BC0', lw=2,
             label=f'h1 = relu({W1[0,0]}·x1 + {W1[0,1]}·x2 + {b1[0]})')
axes[2].plot(x_grid, relu(W1[1, 0] * x_grid + b1[1]), '#FFA726', lw=2,
             label=f'h2 = relu({W1[1,0]}·x1 + {W1[1,1]}·x2 + {b1[1]})')
axes[2].set_title('What each hidden unit computes\n(treating x2=0 for illustration)', fontsize=10)
axes[2].set_xlabel('x1 value')
axes[2].set_ylabel('Hidden unit output')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()
print("The hidden layer re-maps the inputs into a space where they BECOME linearly separable.")
print("This is what hidden layers do: they invent new features that make the problem easier.")

## Section 6 — Backward Pass: How the Chain Rule Carries the Error Back

### The problem with early layers

When we have:
```
x  →  [W₁]  →  h  →  [W₂]  →  ŷ  →  Loss
```

Computing the gradient for **W₂** is easy — it is the last layer:
```
∂Loss/∂W₂ = ∂Loss/∂ŷ × ∂ŷ/∂W₂ = error × h
```
(Same as linear regression, but using `h` as the input instead of `x`)

Computing the gradient for **W₁** is harder — it is two steps away from the loss.
This is where the **chain rule** comes in:

```
∂Loss/∂W₁ = ∂Loss/∂ŷ × ∂ŷ/∂h × ∂h/∂W₁
```

Breaking this down:
- `∂Loss/∂ŷ` = the error (how wrong is the final prediction?)
- `∂ŷ/∂h` = W₂ (how much does the output change per unit change in h?)
- `∂h/∂W₁` = relu'(z₁) × x (how much does h change per unit change in W₁?)

`relu'(z) = 1 if z > 0, else 0` — the **gate**: if a hidden unit was inactive (output=0),
its weights receive zero gradient. They are "frozen" for that observation.

### In plain English

The error flows backward through the network:
1. "The final prediction was too high by $50"
2. "W₂ passed this information through h₂ with weight 0.8 — so h₂ contributed $40 of the error"
3. "h₂ was active (relu gate open) for this observation"
4. "W₁ set h₂ using feature x₃ with weight 1.2 — so W₁₃ contributed proportionally"
5. "Update W₁₃ accordingly"

The signal gets multiplied by each intermediate weight as it flows backward.
Layers far from the output receive a **diluted** error signal — this is the
**vanishing gradient problem** in very deep networks.

In [ ]:
# Implement backpropagation from scratch — fully transparent

def forward(x, W1, b1, W2, b2):
    z1   = W1 @ x + b1
    h    = relu(z1)
    z2   = (W2 * h).sum() + b2
    yhat = sigmoid(z2)
    return z1, h, z2, yhat

def backward(x, y_true, z1, h, z2, yhat, W1, b1, W2, b2):
    # ── Output layer ──────────────────────────────────────────────────────────
    # Loss = (yhat - y)^2 — using MSE for clarity
    # dLoss/dyhat = 2*(yhat - y)
    # dyhat/dz2   = sigmoid'(z2) = yhat*(1-yhat)
    # dLoss/dz2   = dLoss/dyhat * dyhat/dz2
    error       = yhat - y_true
    d_yhat      = 2 * error
    d_z2        = d_yhat * yhat * (1 - yhat)  # chain: through sigmoid

    # dLoss/dW2 = dLoss/dz2 * dz2/dW2 = d_z2 * h
    grad_W2 = d_z2 * h
    grad_b2 = d_z2

    # ── Hidden layer ──────────────────────────────────────────────────────────
    # dLoss/dh = dLoss/dz2 * dz2/dh = d_z2 * W2
    d_h  = d_z2 * W2

    # dh/dz1 = relu'(z1) = 1 if z1 > 0 else 0
    d_z1 = d_h * (z1 > 0).astype(float)   # gate: zero if neuron was inactive

    # dLoss/dW1 = outer product of d_z1 and x
    grad_W1 = np.outer(d_z1, x)
    grad_b1 = d_z1

    return grad_W1, grad_b1, grad_W2, grad_b2

# Train the network on XOR using backprop
np.random.seed(42)
W1 = np.random.randn(4, 2) * 0.5    # 4 hidden units
b1 = np.zeros(4)
W2 = np.random.randn(4) * 0.5
b2 = 0.0

LR = 0.3
losses = []

for epoch in range(3000):
    total_loss = 0
    gW1 = np.zeros_like(W1); gb1 = np.zeros_like(b1)
    gW2 = np.zeros_like(W2); gb2 = 0.0

    for xi, yi in zip(X_xor, y_xor):
        z1, h, z2, yhat = forward(xi, W1, b1, W2, b2)
        total_loss += (yhat - yi)**2
        dW1, db1, dW2, db2_ = backward(xi, yi, z1, h, z2, yhat, W1, b1, W2, b2)
        gW1 += dW1; gb1 += db1; gW2 += dW2; gb2 += db2_

    W1 -= LR * gW1 / len(X_xor)
    b1 -= LR * gb1 / len(X_xor)
    W2 -= LR * gW2 / len(X_xor)
    b2 -= LR * gb2 / len(X_xor)
    losses.append(total_loss / len(X_xor))

print("After 3000 epochs of backpropagation:")
print(f"{'Input':>10}  {'True y':>7}  {'Predicted':>10}  {'Correct?'}")
print("-" * 40)
for xi, yi in zip(X_xor, y_xor):
    _, _, _, yhat = forward(xi, W1, b1, W2, b2)
    correct = '✓' if round(yhat) == yi else '✗'
    print(f"  {str(xi.astype(int)):>8}  {int(yi):>7}  {yhat:>10.4f}  {correct}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.semilogy(losses, color='steelblue', lw=2)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('MSE Loss (log scale)', fontsize=11)
ax1.set_title('XOR Training Loss\n(log scale — drops sharply as network discovers the rule)', fontsize=11)

# Decision boundary of the trained network
xs_grid = np.linspace(-0.2, 1.2, 200)
ys_grid = np.linspace(-0.2, 1.2, 200)
XG, YG  = np.meshgrid(xs_grid, ys_grid)
Z = np.array([
    forward(np.array([xi, yi]), W1, b1, W2, b2)[3]
    for xi, yi in zip(XG.ravel(), YG.ravel())
]).reshape(XG.shape)

ax2.contourf(XG, YG, Z, levels=20, cmap='RdYlGn', alpha=0.6)
for xi, yi_true in zip(X_xor, y_xor):
    color = '#26A69A' if yi_true == 1 else '#EF5350'
    ax2.scatter(xi[0], xi[1], color=color, s=300, edgecolors='black', zorder=5)
    ax2.text(xi[0]+0.05, xi[1]+0.05, f'y={int(yi_true)}', fontsize=11, fontweight='bold')
ax2.set_title('Learned Decision Boundary\n(green = predict 1, red = predict 0)', fontsize=11)
ax2.set_xlabel('x1', fontsize=11)
ax2.set_ylabel('x2', fontsize=11)

plt.suptitle('Neural Network Learns XOR via Backpropagation', fontsize=12)
plt.tight_layout()
plt.show()
print("The network learned the XOR rule that no linear function can express.")
print("Backpropagation carried the error signal back through both layers to update all weights.")

## Section 7 — What Do Hidden Layers Actually Learn?

### Each hidden unit learns a detector

A hidden unit with incoming weights `w` computes:
```
activation = relu(w₁x₁ + w₂x₂ + w₃x₃ + ...)
```
This fires (outputs > 0) when the weighted combination of inputs crosses a threshold.

Different hidden units fire for different combinations of input features.
In doing so, they learn to detect **intermediate concepts**:

| Domain | Input features | Hidden unit might detect |
|---|---|---|
| Titanic | pclass, sex, age, fare | "wealthy young woman" |
| Image | pixel values | "edge at 45°", "circle shape" |
| Text | word frequencies | "negative sentiment words", "question pattern" |
| Housing | MedInc, Latitude | "high-income coastal district" |

### Why deeper networks learn more abstract features

- **Layer 1**: detects basic patterns in raw features (edges, correlations)
- **Layer 2**: detects combinations of Layer 1 patterns (shapes, compound signals)
- **Layer 3**: detects combinations of Layer 2 patterns (objects, concepts)

Each layer builds on the previous layer's vocabulary of features.
This is why deep networks can represent extremely complex functions with relatively few parameters.

In [ ]:
# Apply a trained 2-layer network to Titanic and visualise hidden unit activations
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# Load Titanic
df_t = sns.load_dataset('titanic').dropna(subset=['survived'])
df_t['age']          = df_t['age'].fillna(df_t['age'].median())
df_t['fare']         = df_t['fare'].fillna(df_t['fare'].median())
df_t['embarked']     = df_t['embarked'].fillna('S')
df_t['sex_enc']      = (df_t['sex'] == 'female').astype(int)
df_t['embarked_enc'] = df_t['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df_t['log_fare']     = np.log1p(df_t['fare'])
df_t['family_size']  = df_t['sibsp'] + df_t['parch'] + 1

FEATURES_T = ['pclass','sex_enc','age','sibsp','parch','log_fare','embarked_enc','family_size']
X_t = df_t[FEATURES_T].values
y_t = df_t['survived'].values.astype(int)

sc_t   = StandardScaler()
X_t_sc = sc_t.fit_transform(X_t)

mlp = MLPClassifier(hidden_layer_sizes=(8, 4), activation='relu',
                    max_iter=2000, random_state=42)
mlp.fit(X_t_sc, y_t)
acc = (mlp.predict(X_t_sc) == y_t).mean()
print(f"MLP training accuracy: {acc*100:.1f}%")

# Extract hidden layer activations
W1_mlp = mlp.coefs_[0]      # shape (8_features, 8_hidden)
b1_mlp = mlp.intercepts_[0]

H1 = np.maximum(0, X_t_sc @ W1_mlp + b1_mlp)  # shape (n, 8)

print()
print("Hidden layer activations — first 8 hidden units")
print("Each column is one hidden unit's response to all passengers")
print(pd.DataFrame(H1[:5].round(2), columns=[f'h{i+1}' for i in range(H1.shape[1])]).to_string())
print()
print("What makes each hidden unit fire (positive weights = feature increases activation):")

In [ ]:
# What does each hidden unit "care about"?
W1_df = pd.DataFrame(W1_mlp, index=FEATURES_T,
                     columns=[f'HiddenUnit_{i+1}' for i in range(W1_mlp.shape[1])])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap: which input features each hidden unit weighs
sns.heatmap(W1_df, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            linewidths=0.5, ax=ax1, cbar_kws={'label': 'Weight magnitude'})
ax1.set_title('Layer 1 Weights\nEach column = one hidden unit\nRed = positive, Blue = negative',
              fontsize=11)
ax1.set_xlabel('Hidden Units', fontsize=10)
ax1.set_ylabel('Input Features', fontsize=10)

# Average activation of each hidden unit, split by survival
survived_mask = y_t == 1
H1_surv = H1[survived_mask].mean(axis=0)
H1_died = H1[~survived_mask].mean(axis=0)

x_pos = np.arange(H1.shape[1])
width = 0.35
ax2.bar(x_pos - width/2, H1_surv, width, color='#26A69A', label='Survived')
ax2.bar(x_pos + width/2, H1_died, width, color='#EF5350',  label='Did not survive')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'h{i+1}' for i in range(H1.shape[1])], fontsize=10)
ax2.set_title('Average Hidden Unit Activation\nby Survival Outcome\n'
              'Units that differ = most discriminative', fontsize=11)
ax2.set_ylabel('Mean activation', fontsize=10)
ax2.legend()

plt.suptitle('What the Hidden Layer Learned — Titanic Neural Network', fontsize=12)
plt.tight_layout()
plt.show()

print("Hidden units that activate differently for survivors vs non-survivors")
print("are the most useful intermediate features the network invented.")

## Summary — The Full Picture of Learning

### For each observation
- Each observation casts a **vote** on the direction of each weight
- Vote size = `error × feature_value` — the lever: larger feature value = larger responsibility
- All observations' votes are averaged to get the gradient step
- Many noisy votes average out to the true signal

### The loss surface
- All possible weight combinations form a surface; the height is the error
- Gradient descent walks downhill on this surface
- **Linear regression**: perfect bowl shape, one global minimum
- **Neural networks**: complex surface with local minima, saddle points, flat regions

### Where linear functions fail
- Cannot capture curves, thresholds, interactions, or XOR
- The limitation is fundamental: any linear combination of features is still linear

### Complex functions = stacked simple operations
- Each layer applies: **linear combination** → **non-linear activation**
- Without activation functions, stacked layers collapse into one linear layer
- Each hidden unit learns to **detect a combination** of input features
- Stacking layers learns increasingly abstract features

### Backpropagation: how early layers learn
- Error flows backward through the network using the **chain rule**
- `∂Loss/∂W₁ = error × W₂ × relu_gate × x`
- The ReLU gate: if a hidden unit was inactive for this observation, its weights do not update
- Deep networks face the **vanishing gradient** problem: the signal weakens with each layer

### The hidden layer invents new features
- Early layers learn basic patterns in the input
- Later layers learn combinations of those patterns
- The final layer maps these learned representations to the target
- This is why neural networks can approximate any function — they learn their own features